In [1]:
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm

tqdm.pandas()

In [2]:
DATASET_PATH = '../dataset/E066/'

In [3]:
def encode_histone_exp(row, histone):
    arr = []
    start = row['start']
    end = row['end']
    gene_id = row.index[0]
    count = 1

    for i in range(start, end, 100):
        histone_df = histone.loc[
                    # The same gene ID
                    (histone.index == gene_id) &
                    (
                        # The window is inside the histone
                        ((histone['chromStart'] <= i) & (histone['chromEnd'] >= i + 100)) |
                        # The start of histone is inside the window
                        ((histone['chromStart'] >= i) & (histone['chromStart'] <= i + 100) & (histone['chromEnd'] >= i + 100)) |
                        # The end of histone is inside the window
                        ((histone['chromStart'] <= i) & (histone['chromEnd'] >= i) & (histone['chromEnd'] <= i + 100))
                    )
                    ]
        size = histone_df.shape[0]
        if (size > 0):
            avg = histone_df['signalValue'].mean()
        else:
            avg = 0.0
        
        # print(f"Bin {count}: {i} to {i + 100}: {histone_df.shape[0]}, average: {avg}")
        arr.append(avg)
        count += 1
    return arr

In [5]:
# Load the gene expression file
print("Loading the gene expression file")

column_names = ['chromosome_name',
                'start',
                'end',
                'gene_id',
                'E066',
                'strand',
                'label',
                'external_gene_name',
                'start_position',
                'end_position',
                'tss' 
                ]

E066_df = pd.read_csv(os.path.join(DATASET_PATH, "E066.bed"), sep="\t", names = column_names)
print(f"E066 file: {E066_df.shape}")

Loading the gene expression file
E066 file: (19645, 11)


In [6]:
E066_df

,chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
0,chrX,99889988,99899988,ENSG00000000003,73.205,-1,1,TSPAN6,99883667,99894988,99894988
1,chrX,99834799,99844799,ENSG00000000005,0.191,1,-1,TNMD,99839799,99854882,99839799
2,chr20,49570092,49580092,ENSG00000000419,52.609,-1,1,DPM1,49551404,49575092,49575092
3,chr1,169858408,169868408,ENSG00000000457,4.733,-1,1,SCYL3,169818772,169863408,169863408
4,chr1,169626245,169636245,ENSG00000000460,0.942,1,-1,C1orf112,169631245,169823221,169631245
...,...,...,...,...,...,...,...,...,...,...,...
19640,chr15,102280913,102290913,ENSG00000259658,0.212,-1,-1,RP11-89K11.1,102277302,102285913,102285913
19641,chr15,97966182,97976182,ENSG00000259664,0.000,-1,-1,CTD-2147F2.2,97913601,97971182,97971182
19642,chr16,33642696,33652696,ENSG00000259680,0.071,-1,-1,RP11-812E19.9,33647044,33647696,33647696
19643,chr14,103584344,103594344,ENSG00000259717,0.000,-1,-1,LINC00677,103587184,103589344,103589344


In [ ]:
E066_df.set_index('gene_id', inplace=True)

In [ ]:
E066_df

In [ ]:
E066_df.info()

In [ ]:
# Load the histone data
print("Loading the histone file")
H3K4me1_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me1_df.csv"))
H3K4me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me3_df.csv"))
H3K9me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K9me3_df.csv"))
H3K27me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K27me3_df.csv"))
H3K36me3_df = pd.read_csv(os.path.join(DATASET_PATH, "E066_H3K36me3_df.csv"))

print(H3K4me1_df.shape)
print(H3K4me3_df.shape)
print(H3K9me3_df.shape)
print(H3K27me3_df.shape)
print(H3K36me3_df.shape)

In [ ]:
H3K4me1_df

In [ ]:
# Set the index
H3K4me1_df.set_index('gene_id', inplace=True)
H3K4me3_df.set_index('gene_id', inplace=True)
H3K9me3_df.set_index('gene_id', inplace=True)
H3K27me3_df.set_index('gene_id', inplace=True)
H3K36me3_df.set_index('gene_id', inplace=True)

In [ ]:
print("Generate histone column")
E066_df.loc[:, 'H3K4me1'] = E066_df.progress_apply(lambda row: encode_histone_exp(row, H3K4me1_df), axis = 1)

# Try with polars

In [9]:
import polars as pl

In [10]:
# Load the gene expression file
schema = pl.Schema({
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'E066': pl.Float64,
        'strand': pl.Int64,
        'label': pl.Int64,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64
})

In [11]:
E066_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066.bed"), 
                      separator="\t", 
                      schema=schema,
                      has_header=False,
                      skip_rows=0)

In [12]:
E066_pl

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.191,1,-1,"""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,-1,1,"""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",0.942,1,-1,"""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…,…
"""chr15""",102280913,102290913,"""ENSG00000259658""",0.212,-1,-1,"""RP11-89K11.1""",102277302,102285913,102285913
"""chr15""",97966182,97976182,"""ENSG00000259664""",0.0,-1,-1,"""CTD-2147F2.2""",97913601,97971182,97971182
"""chr16""",33642696,33652696,"""ENSG00000259680""",0.071,-1,-1,"""RP11-812E19.9""",33647044,33647696,33647696


In [13]:
print("Loading the histone file")
H3K4me1_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me1_df.csv"))

Loading the histone file


In [14]:
H3K4me1_pl

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890246,99891002,"""Rank_68310""",106,""".""",5.92171,10.69175,8.59968,462,2.58,10.14
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49575664,49577113,"""Rank_14""",734,""".""",20.49891,73.43851,67.02312,620,55.72,70.21
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49572952,49574573,"""Rank_188""",564,""".""",18.24062,56.41204,51.06926,350,28.6,44.81
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49571998,49572915,"""Rank_71909""",102,""".""",5.67759,10.27033,8.2097,169,19.06,28.23
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49577186,49577379,"""Rank_159548""",51,""".""",3.58302,5.16452,3.52579,35,70.94,72.87
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,-1,"""LINC00677""",103587184,103589344,103589344,"""chr14""",103585293,103585664,"""Rank_133666""",61,""".""",4.02373,6.15866,4.43337,135,9.49,13.2
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,-1,"""LINC00677""",103587184,103589344,103589344,"""chr14""",103584522,103585231,"""Rank_139790""",58,""".""",3.18223,5.88999,4.18566,143,1.78,8.87
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,-1,"""LINC00677""",103587184,103589344,103589344,"""chr14""",103591686,103592058,"""Rank_145378""",55,""".""",3.67747,5.55802,3.87557,247,73.42,77.14


In [24]:
genes_sample= E066_pl.filter(pl.col("gene_id") == "ENSG00000000419")

In [25]:
genes_sample

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092


In [33]:
H3K4me1_pl.filter(pl.col("gene_id") == "ENSG00000000419")

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49575664,49577113,"""Rank_14""",734,""".""",20.49891,73.43851,67.02312,620,55.72,70.21
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49572952,49574573,"""Rank_188""",564,""".""",18.24062,56.41204,51.06926,350,28.6,44.81
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49571998,49572915,"""Rank_71909""",102,""".""",5.67759,10.27033,8.2097,169,19.06,28.23
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49577186,49577379,"""Rank_159548""",51,""".""",3.58302,5.16452,3.52579,35,70.94,72.87
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49571244,49571632,"""Rank_161879""",51,""".""",3.68107,5.10401,3.47041,307,11.52,15.4
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49577629,49577806,"""Rank_163065""",50,""".""",3.52959,5.07326,3.44426,40,75.37,77.14


In [50]:
genes = genes_sample
histone = H3K4me1_pl

In [51]:
 # Create a sequence of window starts for each gene
genes_with_windows = genes.with_columns([
    pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_starts')
]).explode('window_starts')

In [52]:
genes_with_windows = genes_with_windows.with_columns([
        (pl.col('window_starts') + 100).alias('window_ends')
    ])

In [53]:
genes_with_windows

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570092,49570192
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570192,49570292
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570292,49570392
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570392,49570492
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570492,49570592
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49579592,49579692
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49579692,49579792
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49579792,49579892


In [54]:
# Join genes with histone data
joined = genes_with_windows.join(
    histone,
    left_on='gene_id',
    right_on='gene_id',
    how='left'
)

In [55]:
joined

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,window_starts,window_ends,chromosome_name_right,start_right,end_right,E066_right,strand_right,label_right,external_gene_name_right,start_position_right,end_position_right,tss_right,chrom,chromStart,chromEnd,name,score,strand_peak,signalValue,pValue,qValue,peak,startBucket,endBucket
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64,i64,i64,str,i64,i64,f64,i64,i64,str,i64,i64,i64,str,i64,i64,str,i64,str,f64,f64,f64,i64,f64,f64
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570092,49570192,"""chr20""",49570092,49580092,52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49575664,49577113,"""Rank_14""",734,""".""",20.49891,73.43851,67.02312,620,55.72,70.21
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570092,49570192,"""chr20""",49570092,49580092,52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49572952,49574573,"""Rank_188""",564,""".""",18.24062,56.41204,51.06926,350,28.6,44.81
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570092,49570192,"""chr20""",49570092,49580092,52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49571998,49572915,"""Rank_71909""",102,""".""",5.67759,10.27033,8.2097,169,19.06,28.23
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570092,49570192,"""chr20""",49570092,49580092,52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49577186,49577379,"""Rank_159548""",51,""".""",3.58302,5.16452,3.52579,35,70.94,72.87
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49570092,49570192,"""chr20""",49570092,49580092,52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49571244,49571632,"""Rank_161879""",51,""".""",3.68107,5.10401,3.47041,307,11.52,15.4
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49579992,49580092,"""chr20""",49570092,49580092,52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49572952,49574573,"""Rank_188""",564,""".""",18.24062,56.41204,51.06926,350,28.6,44.81
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49579992,49580092,"""chr20""",49570092,49580092,52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49571998,49572915,"""Rank_71909""",102,""".""",5.67759,10.27033,8.2097,169,19.06,28.23
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092,49579992,49580092,"""chr20""",49570092,49580092,52.609,-1,1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49577186,49577379,"""Rank_159548""",51,""".""",3.58302,5.16452,3.52579,35,70.94,72.87


In [ ]:
# Fill missing values with 0.0
joined = joined.with_columns([
    pl.col('signalValue').fill_null(0.0),
    pl.col('chromStart').fill_null(pl.col('window_starts')),
    pl.col('chromEnd').fill_null(pl.col('window_ends'))
])

In [ ]:
joined

In [41]:
# Filter and calculate average signal value
result = joined.filter(
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') >= pl.col('window_starts')) & 
    (pl.col('chromStart') <= pl.col('window_ends')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_starts')) & 
    (pl.col('chromEnd') <= pl.col('window_ends'))
).group_by(['gene_id', 'window_starts']).agg([
    pl.col('signalValue').mean().alias('avg_signal')
]).sort(['gene_id', 'window_starts'])

In [42]:
result

gene_id,window_starts,avg_signal
str,i64,f64
"""ENSG00000000419""",49571192,3.68107
"""ENSG00000000419""",49571292,3.68107
"""ENSG00000000419""",49571392,3.68107
"""ENSG00000000419""",49571492,3.68107
"""ENSG00000000419""",49571592,3.68107
…,…,…
"""ENSG00000000419""",49577192,3.58302
"""ENSG00000000419""",49577292,3.58302
"""ENSG00000000419""",49577592,3.52959


In [43]:
# Fill missing values with 0.0
result = result.with_columns([
    pl.col('avg_signal').fill_null(0.0)
])

In [44]:
result

gene_id,window_starts,avg_signal
str,i64,f64
"""ENSG00000000419""",49571192,3.68107
"""ENSG00000000419""",49571292,3.68107
"""ENSG00000000419""",49571392,3.68107
"""ENSG00000000419""",49571492,3.68107
"""ENSG00000000419""",49571592,3.68107
…,…,…
"""ENSG00000000419""",49577192,3.58302
"""ENSG00000000419""",49577292,3.58302
"""ENSG00000000419""",49577592,3.52959


In [45]:
final_result = result.group_by('gene_id').agg(
    pl.col('avg_signal')
)

In [46]:
final_result

gene_id,avg_signal
str,list[f64]
"""ENSG00000000419""","[3.68107, 3.68107, … 3.52959]"


# Check the Parquet File

This data was from previous step

In [59]:
E066_df = pd.read_parquet(os.path.join(DATASET_PATH, "E066_with_histone.parquet"))

In [57]:
E066_df.head()

,chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,H3K4me1,H3K4me3,H3K9me3,H3K27me3,H3K36me3
0,chrX,99889988,99899988,ENSG00000000003,73.205,-1,1,TSPAN6,99883667,99894988,99894988,"[0.0, 0.0, 5.92171, 5.92171, 5.92171, 5.92171,...","[0.0, 4.19024, 4.19024, 4.19024, 4.19024, 4.19...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[3.39243, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0..."
1,chrX,99834799,99844799,ENSG00000000005,0.191,1,-1,TNMD,99839799,99854882,99839799,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.337...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,chr20,49570092,49580092,ENSG00000000419,52.609,-1,1,DPM1,49551404,49575092,49575092,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[10.55422, 10.55422, 10.55422, 10.55422, 10.55..."
3,chr1,169858408,169868408,ENSG00000000457,4.733,-1,1,SCYL3,169818772,169863408,169863408,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.21981, 4.2198...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,chr1,169626245,169636245,ENSG00000000460,0.942,1,-1,C1orf112,169631245,169823221,169631245,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[3.0104, 3.0104, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [58]:
E066_df.shape

(19645, 16)